# Event Management Portal

A console-based portal where **admins** create and manage events and **students** browse, search, and register for them.

**Main features**
- Role-based login (Admin / Student) with automatic account creation for new usernames
- Add, view and search events
- Attendee registration with capacity and duplicate checks
- Persistent storage in events.json
- Summary report exported to report.csv

## Portal Code

### 1.Imports and global data
- JSON is used to save/load events and CSV is used to export the report.
- EVENTS_FILE is the storage file name, and USERS_DATABASE is an in-memory dictionary (role -> username -> password) that is filled as users log in.

In [1]:
import json
import csv

EVENTS_FILE = "events.json"
USERS_DATABASE = {
    "admin": {},
    "student": {}
}

### 2. File handling functions 
- load_events() reads the saved events and safely returns an empty list if the file is missing or corrupted. 
- save_events() writes all events and their attendees back to the file and reports a permission problem instead of crashing.

In [2]:
def load_events():
    try:
        with open(EVENTS_FILE, "r") as events_file:
            return json.load(events_file)
    except (FileNotFoundError, json.JSONDecodeError):
        return []


def save_events(events):
    try:
        with open(EVENTS_FILE, "w") as events_file:
            json.dump(events, events_file, indent=4)
    except PermissionError:
        print("Error: cannot write to file.")

### 3.Login function
Accepts any username and password. If the username is new for the chosen role, it is registered automatically; if it already exists, the password must match. Returns "admin", "attendee" or "None" (wrong password).

In [3]:
def user_login(role_choice, username, password):
    role_key = "admin" if role_choice == "1" else "student"
    role_users = USERS_DATABASE[role_key]
    
    if username not in role_users:
        role_users[username] = password
        print(f"\n[System: New {role_key} account '{username}' created dynamically!]")
        return "admin" if role_key == "admin" else "attendee"

    if role_users[username] == password:
        return "admin" if role_key == "admin" else "attendee"

    return None


### 4.Event record functions (backend)
- add_record() creates a new event with an auto-generated ID and an empty attendee list.
- search_record() returns every event whose name contains the keyword (case-insensitive).
- show_events() prints events with the number of seats left.

In [ ]:
def add_record(events, event_name, event_date, event_capacity):
    
    new_event = {"id": len(events) + 1, "name": event_name, "date": event_date,
                 "capacity": event_capacity, "attendees": []}
    events.append(new_event)
    return new_event


def search_record(events, keyword):
    
    matching_events = []
    for event in events:
        if keyword.lower() in event["name"].lower():
            matching_events.append(event)
    return matching_events


def show_events(events):
    
    if not events:
        print("No events found.")
    for event in events:
        seats_left = event["capacity"] - len(event["attendees"])
        print(f"{event['id']}. {event['name']} | {event['date']} | Seats left: {seats_left}")

### 5.Registration function 
Registers a person for an event after checking that the event exists, the person is not already registered, and seats are still available and returns a message describing the result.

In [ ]:
def register_attendee(events, event_id, attendee_name):
    
    for event in events:
        if event["id"] == event_id:
            if attendee_name in event["attendees"]:
                return "You are already registered."
            if len(event["attendees"]) >= event["capacity"]:
                return "Sorry, this event is full."
            event["attendees"].append(attendee_name)
            return f"Registered! Confirmation sent to {attendee_name}."
    return "Event not found."

### 6.Report function
Writes a summary of all events to report.csv (Event, Date, Capacity, Registered) and also prints it on screen. A PermissionError (for example, file open in Excel) is handled.

In [6]:
def generate_report(events):
    
    try:
        with open("report.csv", "w", newline="") as report_file:
            csv_writer = csv.writer(report_file)
            csv_writer.writerow(["Event", "Date", "Capacity", "Registered"])
            for event in events:
                csv_writer.writerow([event["name"], event["date"], event["capacity"], len(event["attendees"])])
                print(f"{event['name']}: {len(event['attendees'])}/{event['capacity']} registered")
        print("Report saved to report.csv")
    except PermissionError:
        print("Error: cannot write report.csv (is it open?)")

### 7.Menus (user interface)
- admin_menu() lets admins add events, view, search, generate reports and log out. 
- attendee_menu() lets students view, search, register and log out.
- Both validate numeric input with try/except ValueError and reject invalid menu choices with if/else.

In [7]:
def admin_menu(events):
    while True:
        print("\n1. Add event  2. View events  3. Search  4. Report  5. Logout")
        menu_choice = input("Choose: ")
        if menu_choice == "1":
            try:
                event_name = input("Event name: ")
                event_date = input("Date: ")
                event_capacity = int(input("Capacity: "))
                add_record(events, event_name, event_date, event_capacity)
                save_events(events)
                print("Event added.")
            except ValueError:
                print("Capacity must be a number.")
        elif menu_choice == "2":
            show_events(events)
        elif menu_choice == "3":
            show_events(search_record(events, input("Keyword: ")))
        elif menu_choice == "4":
            generate_report(events)
        elif menu_choice == "5":
            break
        else:
            print("Invalid choice.")


def attendee_menu(events):
    while True:
        print("\n1. View events  2. Search  3. Register  4. Logout")
        menu_choice = input("Choose: ")
        if menu_choice == "1":
            show_events(events)
        elif menu_choice == "2":
            show_events(search_record(events, input("Keyword: ")))
        elif menu_choice == "3":
            try:
                event_id = int(input("Event ID: "))
                attendee_name = input("Your name: ")
                print(register_attendee(events, event_id, attendee_name))
                save_events(events)
            except ValueError:
                print("Event ID must be a number.")
        elif menu_choice == "4":
            break
        else:
            print("Invalid choice.")

### 8.Main program
Loads saved events, shows the portal's main menu, asks for role, username and password, then opens the admin or attendee menu depending on the result of user_login()

In [8]:
def main():
    events = load_events()
    while True:
        print("\n=== EVENT PORTAL ===\n1. Login\n2. Exit")
        main_menu_choice = input("Choose: ")
        if main_menu_choice == "1":
            print("\nIdentify your role:\n1. Admin\n2. Student")
            role_choice = input("Choose (1 or 2): ")
            if role_choice in ["1", "2"]:
                username = input("Username: ")
                password = input("Password: ")
                user_role = user_login(role_choice, username, password)

                if user_role == "admin":
                    admin_menu(events)
                elif user_role == "attendee":
                    attendee_menu(events)
                else:
                    print("Incorrect password for this username.")
            else:
                print("Invalid role selection.")
        elif main_menu_choice == "2":
            print("Goodbye!")
            break
        else:
            print("Invalid choice.")

In [9]:
if __name__ == "__main__":
    main()


=== EVENT PORTAL ===
1. Login
2. Exit


Choose:  1



Identify your role:
1. Admin
2. Student


Choose (1 or 2):  1
Username:  areesha
Password:  areesha123



[System: New admin account 'areesha' created dynamically!]

1. Add event  2. View events  3. Search  4. Report  5. Logout


Choose:  1
Event name:  hackathon
Date:  30-11-2026
Capacity:  3000


... (27 outputs left out)

Choose:  5



=== EVENT PORTAL ===
1. Login
2. Exit


Choose:  2


Goodbye!
